# LLama 3.1 (Decoder) based distributed training

70p-30np Distribution



## Total Object Models: 9


## Training

1.	Camping
2.	Customer_Order
3.	Ecommerce
4.	Onlinestore
5.	Decider
6.  Library OM
7.  CSOS
8.  Flagship

## Testing

9.	Bank



--------------------------------

## Total Training Data: 13236 (100%)
--------------------------------

### Training set P :  9265 (70% of Training Data)

### Training set NP : 3971 (30% of Training Data)

-----------------------------
## Total Testing Data: 32 (100% of Total Data)
----------------------------

### Testing set P : 10 (31% of Testing Data)

### Testing set NP : 22 (69%% of Testing Data)

In [1]:
# Install necessary libraries
!pip install transformers datasets accelerate bitsandbytes peft pandas openpyxl scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!pip install -U bitsandbytes

In [3]:
# Import necessary libraries
import os
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, BitsAndBytesConfig
from datasets import Dataset, DatasetDict
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score

In [4]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [5]:
# Disable W&B logging if not needed
os.environ["WANDB_DISABLED"] = "true"

In [6]:
# Define your Hugging Face access token
HF_ACCESS_TOKEN = ""

import os
os.environ["HF_TOKEN"] = ""

In [7]:
!pip install -U bitsandbytes

In [8]:
# Load the LLaMA 3.1 8B model and tokenizer with the access token
model_name = "meta-llama/Meta-Llama-3-8B"

# Configure quantization
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,  # Use 8-bit quantization
    llm_int8_enable_fp32_cpu_offload=True  # Enable CPU offloading for FP32 layers
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_ACCESS_TOKEN)

# Add a padding token if not present
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})

# Load model with quantization
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
    token=HF_ACCESS_TOKEN
)

# Resize model embeddings to account for the new padding token
model.resize_token_embeddings(len(tokenizer))

# Configure LoRA for fine-tuning
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none"
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/177 [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [12]:
# Load and preprocess the training dataset
def load_training_data(file_path):
    df = pd.read_csv(file_path)
    return df

# Preprocess function to tokenize input and output
def preprocess_function(examples):
    inputs = tokenizer(examples["OM_Regular"], truncation=True, padding="max_length", max_length=256)
    outputs = tokenizer(examples["OM_Prediction"], truncation=True, padding="max_length", max_length=256)
    inputs["labels"] = outputs["input_ids"]
    return inputs

# Load training data
train_df = load_training_data("raw_8_om_training_set.csv")
train_dataset = Dataset.from_pandas(train_df)

# Tokenize the training data
tokenized_train_dataset = train_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/13335 [00:00<?, ? examples/s]

In [14]:
# Split the training data into training and validation sets
train_test_split = tokenized_train_dataset.train_test_split(test_size=0.2, seed=42)
dataset = DatasetDict({
    "train": train_test_split["train"],
    "validation": train_test_split["test"]
})

print(f"Training samples: {len(dataset['train'])}")
print(f"Validation samples: {len(dataset['validation'])}")

Training samples: 10668
Validation samples: 2667


In [ ]:
# Define metrics for evaluation
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Convert logits to predictions and flatten from (batch, seq_length) to (batch*seq_length,)
    predictions = torch.argmax(torch.tensor(logits), dim=-1).numpy().flatten()
    labels = labels.flatten()

    # Remove ignored tokens (-100 is typically used for ignored tokens)
    valid_indices = labels != -100
    predictions = predictions[valid_indices]
    labels = labels[valid_indices]

    accuracy = accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, average="weighted", zero_division=0)
    recall = recall_score(labels, predictions, average="weighted", zero_division=0)
    f1 = f1_score(labels, predictions, average="weighted", zero_division=0)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
# Set up training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=1,  # Reduce batch size to avoid OOM
    gradient_accumulation_steps=6,  # Use gradient accumulation to simulate a larger batch size
    per_device_eval_batch_size=2,
    num_train_epochs=8,
    weight_decay=0.01,
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=8,
    fp16=True,  # Enable mixed precision
    push_to_hub=False,
)

print("Training arguments set up successfully.")

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Training arguments set up successfully.


In [ ]:
# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

print("Trainer initialized successfully.")

<ipython-input-11-6844c528c354>:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Trainer initialized successfully.


In [ ]:
# Fine-tune the model
print("Starting fine-tuning...")
trainer.train()
print("Fine-tuning completed.")

Starting fine-tuning...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.821000,0.666974,0.221195,0.233924,0.221195,0.216810
2,0.718500,0.620658,0.264922,0.264379,0.264922,0.263230
3,0.838700,0.636344,0.240164,0.244628,0.240164,0.237448
4,0.715600,0.709210,0.135089,0.146075,0.135089,0.123858
5,0.635600,0.606327,0.266328,0.265837,0.266328,0.265042
6,0.596400,0.595032,0.267452,0.266647,0.267452,0.267041
7,0.697900,0.595688,0.267424,0.266634,0.267424,0.267022


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:260: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:260: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:260: U

Fine-tuning completed.


In [ ]:
# Evaluate the model on the validation set
print("Evaluating on validation set...")
validation_results = trainer.evaluate()
print("Validation Results:", validation_results)

Evaluating on validation set...


Validation Results: {'eval_loss': 0.5956884026527405, 'eval_accuracy': 0.2674235611510791, 'eval_precision': 0.2666344744637439, 'eval_recall': 0.2674235611510791, 'eval_f1': 0.2670217844134829, 'eval_runtime': 26.8609, 'eval_samples_per_second': 5.175, 'eval_steps_per_second': 2.606, 'epoch': 7.92057761732852}


In [ ]:
# Load and preprocess the test dataset
def load_test_data(file_path):
    df = pd.read_excel(file_path)
    return df

# Preprocess function for test data
def preprocess_test_function(examples):
    # Tokenize only the input text column
    inputs = tokenizer(examples["OM_Regular"], truncation=True, padding="max_length", max_length=512)
    # Keep the raw label for later reference (do not tokenize it)
    inputs["raw_labels"] = examples["OM_Prediction"]
    return inputs

# Load test data
test_df = load_test_data("raw_testset_bank.xlsx")
test_dataset = Dataset.from_pandas(test_df)

# Tokenize the test data
tokenized_test_dataset = test_dataset.map(preprocess_test_function, batched=True)

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

In [ ]:
# Perform inference on the test set
print("Performing inference on test set...")
predictions = trainer.predict(tokenized_test_dataset)

# Decode predictions
predicted_labels = tokenizer.batch_decode(torch.argmax(torch.tensor(predictions.predictions), dim=-1), skip_special_tokens=True)

print("Inference completed.")

Performing inference on test set...


Inference completed.


In [10]:
# Save predictions
test_texts = test_df["OM_Regular"].tolist()
true_labels = test_df["OM_Prediction"].tolist()

results_df = pd.DataFrame({
    "Text": test_texts,
    "True_Label": true_labels,
    "Predicted_Label": predicted_labels
})

# Save as CSV
results_df.to_csv("raw_testset_bank_pred_1.csv", index=False)

# Save as Excel
results_df.to_excel("raw_testset_bank_pred_1.xlsx", index=False)

print("Test results saved to both CSV and Excel files.")

Test results saved to both CSV and Excel files.


# Calculating Results from unseen Testset

In [11]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_curve, roc_auc_score
from sklearn.metrics import precision_recall_curve, classification_report

In [12]:
dc = pd.read_excel('raw_testset_bank.xlsx')

In [13]:
X_test2 = dc['OM_Regular'].values
y_test2 = dc['OM_Prediction'].values

In [14]:
print(X_test2.shape)
print(y_test2.shape)

print("X data type: ", X_test2.dtype)
print("y data type: ", y_test2.dtype)

(32,)
(32,)
X data type:  object
y data type:  int64


In [15]:
print(y_test2)

[1 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 1 1 1 1 1]


In [22]:
dd = pd.read_excel('raw_testset_bank_pred_1.xlsx')

In [23]:
X_test_pred2 = dd['OM_Regular'].values
y_test_pred2 = dd['OM_Prediction'].values

In [24]:
print (y_test_pred2 )

[0 0 0 1 0 0 0 0 0 0 0 0 1 1 0 1 1 1 1 1 1 0 0 0 1 0 1 1 0 1 1 1]


In [25]:
precision = precision_score(y_test2, y_test_pred2)
print("Testing: Precision = %f" % precision)


recall = recall_score(y_test2, y_test_pred2)
print("Testing: Recall = %f" % recall)


f1 = f1_score(y_test2, y_test_pred2)
print("Testing: F1 Score = %f" % f1)

print("\nConfusion Matrix (Test Data):\n", confusion_matrix(y_test2, y_test_pred2))

Testing: Precision = 0.466667
Testing: Recall = 0.700000
Testing: F1 Score = 0.560000

Confusion Matrix (Test Data):
 [[14  8]
 [ 3  7]]


In [26]:
print(classification_report(y_test2,y_test_pred2))

              precision    recall  f1-score   support

           0       0.82      0.64      0.72        22
           1       0.47      0.70      0.56        10

    accuracy                           0.66        32
   macro avg       0.65      0.67      0.64        32
weighted avg       0.71      0.66      0.67        32

